# Auxiliary Pipeline

Thin driver notebook for charts, DOCX export, spellcheck, polls, avatar scripts, and HeyGen operations.

In [ ]:
from sea_pipeline.shared.config import PipelineConfig
from sea_pipeline.shared.logging import print_stage_result
from sea_pipeline.auxiliary.runner import list_auxiliary_stages, run_auxiliary_stage

config = PipelineConfig.from_repo_root()
list_auxiliary_stages()

## Validate Environment

In [ ]:
result = run_auxiliary_stage(config, "validate_env")
print_stage_result(result)

## Charts

In [ ]:
result = run_auxiliary_stage(config, "charts_extract_and_merge")
print_stage_result(result)

In [ ]:
result = run_auxiliary_stage(config, "charts_render")
print_stage_result(result)

## DOCX Exports

In [ ]:
result = run_auxiliary_stage(config, "docx_export_text")
print_stage_result(result)

In [ ]:
result = run_auxiliary_stage(config, "docx_export_with_images")
print_stage_result(result)

## Spellcheck And Indicator Links

In [ ]:
result = run_auxiliary_stage(config, "spellcheck_audit")
print_stage_result(result)

In [ ]:
result = run_auxiliary_stage(config, "indicator_links")
print_stage_result(result)

## Poll Workflows

In [ ]:
result = run_auxiliary_stage(config, "polls_generate")
print_stage_result(result)

In [ ]:
result = run_auxiliary_stage(config, "polls_insert")
print_stage_result(result)

## Avatar Scripts And HeyGen Operations

In [ ]:
result = run_auxiliary_stage(config, "avatar_script_generation")
print_stage_result(result)

In [ ]:
result = run_auxiliary_stage(config, "generate_payloads")
print_stage_result(result)

In [ ]:
result = run_auxiliary_stage(config, "generate_videos")
print_stage_result(result)

In [ ]:
result = run_auxiliary_stage(config, "list_blob_clips")
print_stage_result(result)

In [ ]:
result = run_auxiliary_stage(config, "download_blob_clips")
print_stage_result(result)

In [ ]:
result = run_auxiliary_stage(config, "add_backgrounds")
print_stage_result(result)

## Photo Caption Matcher

Match lesson photo assets against `02_Inputs/piwigo.csv` using a hybrid matcher: hash for shortlist selection, then AKAZE feature matching for final verification.

Included: regular lesson photos.
Skipped: infographics, learning objectives, key takeaways, module/chapter/lesson covers, resources, avatars, videos, the first header image in each lesson, images inside connection templates, and lesson frame assets like `lesson111_frame.webp`.

`SKIP_UNKNOWNS = True` means process only blank captions.
`SKIP_UNKNOWNS = False` means process blank captions and captions currently set to `"unknown"`.

Because the Azure image is the cropped version of the Piwigo image, the hash settings now only control shortlist generation. The actual acceptance decision comes from AKAZE feature matching on the shortlisted Piwigo candidates.

`MATCH_BIT_ERROR_RATE` and `MIN_CANDIDATE_COVERAGE` affect which Piwigo images make it into the shortlist.
`AKAZE_MIN_GOOD_MATCHES`, `AKAZE_MIN_INLIERS`, `AKAZE_MIN_INLIER_RATIO`, and `AKAZE_MIN_CANDIDATE_COVERAGE` control the final exact-match decision.
Start with the defaults below before loosening anything.

Set `PIWIGO_BASE_URL` to your Piwigo host or even a sample picture URL. The notebook normalizes either form and resolves the real top-candidate image via the Piwigo API. Add `PIWIGO_USERNAME` and `PIWIGO_PASSWORD` only if the gallery requires login.

`TEST_RUN = True` keeps the run bounded by lesson count, while `IMAGE_LIMIT` caps how many eligible images are actually processed.


In [ ]:
from functools import lru_cache
from html import escape

from IPython.display import HTML, display

from sea_pipeline.auxiliary.photo_caption_matcher import (
    create_piwigo_session,
    fetch_piwigo_image_info,
    update_photo_captions_from_piwigo,
)

LESSON_ROOT = config.paths.outputs_dir / "SEA_Modules" / "en"
PIWIGO_CSV_PATH = config.paths.inputs_dir / "piwigo.csv"
LESSON_GLOB = "Module_*/*.json"
SKIP_UNKNOWNS = False
TEST_RUN = True
SAMPLE_SIZE = 40
IMAGE_LIMIT = 20

MATCH_STRATEGY = "hybrid_akaze"
MATCH_BIT_ERROR_RATE = 0.18
MIN_CANDIDATE_COVERAGE = 0.60
PIWIGO_HASH_SHORTLIST_SIZE = 12

AKAZE_MIN_GOOD_MATCHES = 12
AKAZE_MIN_INLIERS = 8
AKAZE_MIN_INLIER_RATIO = 0.30
AKAZE_MIN_CANDIDATE_COVERAGE = 0.08

PIWIGO_BASE_URL = "https://seh-sea-piwigo.azurewebsites.net/picture.php?/5007"
PIWIGO_CATEGORY_ID = 209
PIWIGO_USERNAME = None
PIWIGO_PASSWORD = None

PIWIGO_SESSION = (
    create_piwigo_session(PIWIGO_BASE_URL, username=PIWIGO_USERNAME, password=PIWIGO_PASSWORD)
    if PIWIGO_BASE_URL else None
)

@lru_cache(maxsize=2048)
def build_piwigo_preview_url(photo_id: str) -> str | None:
    if not PIWIGO_BASE_URL or not photo_id or photo_id == "unknown":
        return None
    try:
        info = fetch_piwigo_image_info(PIWIGO_BASE_URL, photo_id, session=PIWIGO_SESSION)
    except Exception as exc:
        print(f"     piwigo_preview_error={photo_id} | {type(exc).__name__}: {exc}")
        return None
    return info.image_url or info.page_url

def show_photo_caption_match_preview(event: dict) -> None:
    best_metrics = event.get("best_metrics") or {}
    lesson_src = event.get("src", "")
    best_photo_id = str(event.get("best_photo_id") or "unknown")
    best_piwigo_src = event.get("best_piwigo_src")
    lesson_path = escape(str(event.get("lesson_path", "")))
    image_path = escape(str(event.get("image_path", "")))
    match_status = escape(str(event.get("match_status", "")))
    metric_parts = []
    if best_metrics.get("candidate_coverage") is not None:
        metric_parts.append(f"coverage={best_metrics["candidate_coverage"]:.2f}")
    if best_metrics.get("matched_regions") is not None and best_metrics.get("candidate_regions") is not None:
        metric_parts.append(f"regions={best_metrics["matched_regions"]}/{best_metrics["candidate_regions"]}")
    if best_metrics.get("avg_bit_error") is not None:
        metric_parts.append(f"avg_bit_error={best_metrics["avg_bit_error"]:.3f}")
    if best_metrics.get("good_matches") is not None:
        metric_parts.append(f"good_matches={best_metrics["good_matches"]}")
    if best_metrics.get("inliers") is not None:
        metric_parts.append(f"inliers={best_metrics["inliers"]}")
    if best_metrics.get("inlier_ratio") is not None:
        metric_parts.append(f"inlier_ratio={best_metrics["inlier_ratio"]:.2f}")
    if best_metrics.get("mean_distance") is not None and best_metrics.get("mean_distance") != float("inf"):
        metric_parts.append(f"mean_distance={best_metrics["mean_distance"]:.2f}")
    if best_metrics.get("score") is not None:
        metric_parts.append(f"score={best_metrics["score"]:.2f}")
    metrics_html = escape(" | ".join(metric_parts)) if metric_parts else "no metrics"
    right_panel = (
        f'<img src="{escape(best_piwigo_src)}" style="max-width:100%; max-height:260px; object-fit:contain; border:1px solid #d8e2dc; border-radius:8px; background:#fff;" />'
        if best_piwigo_src
        else f'<div style="height:260px; display:flex; align-items:center; justify-content:center; border:1px dashed #b8c5bf; border-radius:8px; background:#f8faf8; color:#4f5d53; padding:12px; text-align:center;">Top Piwigo candidate ID: {escape(best_photo_id)}<br><span style="font-size:12px;">Could not resolve the image URL from Piwigo.</span></div>'
    )
    html = f"""
<div style="display:grid; grid-template-columns:1fr 1fr; gap:16px; margin:12px 0 18px; align-items:start;">
  <div>
    <div style="font:600 13px/1.4 sans-serif; margin-bottom:6px; color:#253128;">Lesson image</div>
    <img src="{escape(lesson_src)}" style="max-width:100%; max-height:260px; object-fit:contain; border:1px solid #d8e2dc; border-radius:8px; background:#fff;" />
  </div>
  <div>
    <div style="font:600 13px/1.4 sans-serif; margin-bottom:6px; color:#253128;">Top Piwigo candidate ({escape(best_photo_id)})</div>
    {right_panel}
  </div>
</div>
<div style="font:12px/1.5 monospace; color:#33413a; margin:-4px 0 14px;">
  lesson={lesson_path} | image={image_path} | status={match_status} | {metrics_html}
</div>
"""
    display(HTML(html))

# imagehash and OpenCV are required at runtime for this tool.
# Install once if needed: pip3 install ImageHash opencv-python
LESSON_ROOT, PIWIGO_CSV_PATH


In [ ]:
photo_caption_result = update_photo_captions_from_piwigo(
    LESSON_ROOT,
    PIWIGO_CSV_PATH,
    lesson_glob=LESSON_GLOB,
    skip_unknowns=SKIP_UNKNOWNS,
    test_run=TEST_RUN,
    sample_size=SAMPLE_SIZE,
    image_limit=IMAGE_LIMIT,
    match_strategy=MATCH_STRATEGY,
    match_bit_error_rate=MATCH_BIT_ERROR_RATE,
    min_candidate_coverage=MIN_CANDIDATE_COVERAGE,
    piwigo_base_url=PIWIGO_BASE_URL,
    piwigo_category_id=PIWIGO_CATEGORY_ID,
    piwigo_username=PIWIGO_USERNAME,
    piwigo_password=PIWIGO_PASSWORD,
    piwigo_hash_shortlist_size=PIWIGO_HASH_SHORTLIST_SIZE,
    akaze_min_good_matches=AKAZE_MIN_GOOD_MATCHES,
    akaze_min_inliers=AKAZE_MIN_INLIERS,
    akaze_min_inlier_ratio=AKAZE_MIN_INLIER_RATIO,
    akaze_min_candidate_coverage=AKAZE_MIN_CANDIDATE_COVERAGE,
    piwigo_image_url_builder=build_piwigo_preview_url,
    match_reporter=show_photo_caption_match_preview,
    log=print,
)

{k: photo_caption_result[k] for k in [
    "status", "test_run", "sample_size", "image_limit", "match_strategy", "match_bit_error_rate", "min_candidate_coverage",
    "piwigo_hash_shortlist_size", "akaze_min_good_matches", "akaze_min_inliers", "akaze_min_inlier_ratio", "akaze_min_candidate_coverage",
    "stopped_after_image_limit", "skip_unknowns", "scanned_lessons", "scanned_images", "changed_lessons", "matched", "unknown", "ambiguous", "errors"
]}


In [ ]:
photo_caption_result["changes"][:10]


## Resource Image WebP Backfill

Scan the English lesson JSONs for resource image URLs under `images/Resources`, check whether each expected `.webp` exists in Azure, and if not, check for a sibling `.png`. When only the `.png` exists, the stage downloads it, converts it to `.webp`, and uploads the `.webp` back into the same folder.

`RESOURCE_IMAGE_BACKFILL_LIMIT = None` scans the full output set. Set an integer to test on a smaller subset of resource URLs first.


In [ ]:
RESOURCE_IMAGE_BACKFILL_LESSON_GLOB = "Module_*/*.json"
RESOURCE_IMAGE_BACKFILL_LIMIT = None
RESOURCE_IMAGE_BACKFILL_OVERWRITE = True

config.stage_options["resource_image_backfill_lesson_glob"] = RESOURCE_IMAGE_BACKFILL_LESSON_GLOB
config.stage_options["resource_image_backfill_limit"] = RESOURCE_IMAGE_BACKFILL_LIMIT
config.stage_options["resource_image_backfill_overwrite"] = RESOURCE_IMAGE_BACKFILL_OVERWRITE

RESOURCE_IMAGE_BACKFILL_LESSON_GLOB, RESOURCE_IMAGE_BACKFILL_LIMIT, RESOURCE_IMAGE_BACKFILL_OVERWRITE


In [ ]:
resource_image_backfill_result = run_auxiliary_stage(config, "resource_image_webp_backfill")
print_stage_result(resource_image_backfill_result)


In [ ]:
resource_image_backfill_result.get("changes", [])[:20]
